Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn
!pip install pandas
!pip install tqdm
!pip install scikit-image
!pip install scipy
!pip install ace_tools

Calling the Libraries:

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
import glob
import os

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs/001/L_Fore/01.bmp'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)


Train

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize

# ----------------------------
# CONFIGURATION
# ----------------------------
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (60, 120)  # Height x Width
PATCH_SIZE = 10
stride = 10
#LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]
LBP_CONFIGS = [(2, 8)]
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
TRAIN_INDICES = [1, 2, 3, 4, 5, 6, 7, 8, 9]  # Protocol 3 - training images

# ----------------------------
# RIU2 MAPPING
# ----------------------------
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        if transitions <= 2:
            table[i] = sum(min_rotation)
        else:
            table[i] = P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# ----------------------------
# LBP FEATURE EXTRACTOR
# ----------------------------
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# ----------------------------
# MAIN LOOP (TRAINING SET - STRATEGY 2: NO FUSION)
# ----------------------------
train_lbp_features = []
train_labels = []

print("\n🔄 Extracting individual finger features from MMCBNU_6000 (Protocol 3 - Strategy 2)...")

subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for finger in FINGER_LIST:
        finger_path = os.path.join(subject_path, finger)
        for img_idx in TRAIN_INDICES:
            img_path = os.path.join(finger_path, f"{img_idx:02d}.bmp")
            print(f"📥 Loading: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"❌ Missing image: {img_path}")
                continue

            img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
            img = cv2.fastNlMeansDenoising(img, h=10)
            img = cv2.equalizeHist(img).astype(np.float64) / 255.0
            img = (img - np.mean(img)) / (np.std(img) + 1e-8)

            feature_vector = []

            for y in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                for x in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                    block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                    block_hist = []
                    for R, P in LBP_CONFIGS:
                        hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                        block_hist.extend(hist)
                    feature_vector.extend(block_hist)

            if len(feature_vector) > 0:
                label = f"{subj}_{finger}_img{img_idx:02d}"
                train_lbp_features.append(feature_vector)
                train_labels.append(label)

# ----------------------------
# FINALIZE FEATURE SET
# ----------------------------
train_lbp_features = np.array(train_lbp_features, dtype=np.float32)
train_lbp_features = normalize(train_lbp_features, norm='l2')
train_labels = np.array(train_labels)

print("\n✅ Feature extraction complete!")
print("🔢 Feature matrix shape:", train_lbp_features.shape)
print("🟢 Example labels:", train_labels[:5])


Test

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize

# === TEST CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (60, 120)
PATCH_SIZE = 10
stride = 10
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
TEST_INDICES = [10]  # Protocol 3 test images
# LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]
LBP_CONFIGS = [(2, 8)]
# === MAPPING FUNCTIONS ===
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        if transitions <= 2:
            table[i] = sum(min_rotation)
        else:
            table[i] = P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === STRATEGY 2: INDIVIDUAL FINGERS (NO FUSION)
test_lbp_features = []
test_labels = []

print("\n🧪 Extracting test features from MMCBNU_6000 (Strategy 2 - Protocol 3)...")

subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for finger in FINGER_LIST:
        finger_path = os.path.join(subject_path, finger)
        for img_idx in TEST_INDICES:
            img_path = os.path.join(finger_path, f"{img_idx:02d}.bmp")
            print(f"📥 Loading: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"❌ Missing image: {img_path}")
                continue

            img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
            img = cv2.fastNlMeansDenoising(img, h=10)
            img = cv2.equalizeHist(img).astype(np.float64) / 255.0
            img = (img - np.mean(img)) / (np.std(img) + 1e-8)

            feature_vector = []

            for y in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                for x in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                    block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                    block_hist = []
                    for R, P in LBP_CONFIGS:
                        hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                        block_hist.extend(hist)
                    feature_vector.extend(block_hist)

            if len(feature_vector) > 0:
                label = f"{subj}_{finger}_img{img_idx:02d}"
                test_lbp_features.append(feature_vector)
                test_labels.append(label)

# === FINALIZE ===
test_lbp_features = np.array(test_lbp_features, dtype=np.float32)
test_lbp_features = normalize(test_lbp_features, norm='l2')
test_labels = np.array(test_labels)

print("\n✅ Test feature extraction complete!")
print("🔢 Test feature matrix shape:", test_lbp_features.shape)
print("🧾 Sample test labels:", test_labels[:5])


Benchmarking

In [ ]:
import numpy as np

# ✅ Helper to extract subject ID and finger from label
def extract_subject_and_finger(label):
    parts = label.split('_')  # Format: subj_finger_imgXX
    return parts[0], parts[1]  # e.g., "008", "L_Fore"

correct_matches = 0
total_tests = len(test_lbp_features)

print("\n🔍 Classifying test data using Manhattan distance (Match: Subject + Finger)...\n")

for i in range(total_tests):
    test_vec = test_lbp_features[i]
    true_label = test_labels[i]

    # 📏 Compute Manhattan distances
    distances = np.sum(np.abs(train_lbp_features - test_vec), axis=1)
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]

    # 🎯 Extract subject and finger
    true_subj, true_finger = extract_subject_and_finger(true_label)
    pred_subj, pred_finger = extract_subject_and_finger(predicted_label)

    # ✅ Match check
    if true_subj == pred_subj and true_finger == pred_finger:
        correct_matches += 1
        match_symbol = "✅"
    else:
        match_symbol = "❌"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {match_symbol}")

# 📊 Final Accuracy
accuracy = (correct_matches / total_tests) * 100
print("\n📊 Final Results")
print(f"✅ Correct subject+finger matches: {correct_matches} / {total_tests}")
print(f"🎯 Recognition Accuracy (Subject + Finger): {accuracy:.2f}%")


Session Independent R5

In [ ]:
import numpy as np

# === Configuration ===
rank_correct = 0
total_tests = len(test_labels)

print("📊 Calculating Session-Independent CMC Rank-1 — Match: Subject + Finger...")

for i in range(total_tests):
    proj_test = test_lbp_features[i]
    test_label = test_labels[i]

    # ✅ Extract subject and finger (ignore session)
    test_parts = test_label.split('_')
    test_subject = test_parts[0]
    test_finger = test_parts[1]
    test_id = f"{test_subject}_{test_finger}"

    # 📏 Compute Manhattan distances
    distances = np.sum(np.abs(train_lbp_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # 🔍 Get the top-1 predicted label
    candidate_label = train_labels[sorted_indices[0]]
    candidate_parts = candidate_label.split('_')
    candidate_subject = candidate_parts[0]
    candidate_finger = candidate_parts[1]
    candidate_id = f"{candidate_subject}_{candidate_finger}"

    # ✅ Match check (subject + finger)
    if candidate_id == test_id:
        rank_correct += 1

# === Final CMC Result
accuracy = (rank_correct / total_tests) * 100
print(f"🎯 Rank-1 Accuracy (Subject + Finger): {accuracy:.2f}%")


Session Independent CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ✅ Helper function: extract subject and finger (ignore session)
def extract_subject_finger(label):
    parts = label.split('_')
    return f"{parts[0]}_{parts[1]}"  # subject_finger

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_lbp_features)

print("📊 Calculating Session-Independent CMC Curve (Matching: Subject + Finger)...")

for i in range(total_tests):
    proj_test = test_lbp_features[i]
    true_label = test_labels[i]
    true_id = extract_subject_finger(true_label)

    # 📏 Compute Manhattan distances to all training samples
    distances = np.sum(np.abs(train_lbp_features - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # ✅ Find the first rank where subject+finger match
    for r in range(max_rank):
        candidate_label = train_labels[sorted_indices[r]]
        candidate_id = extract_subject_finger(candidate_label)

        if candidate_id == true_id:
            rank_correct[r:] += 1
            break

# ✅ Normalize to percentage
cmc_curve = (rank_correct / total_tests) * 100

# ✅ Plotting the CMC curve
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Independent CMC (Subject+Finger)", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("CMC Curve — Subject + Finger Matching")
plt.grid(True)
plt.xticks(np.arange(0, max_rank + 1, 10))
plt.legend()
plt.tight_layout()
plt.show()

# ✅ Print key rank accuracies
print(f"🎯 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🎯 Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🎯 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"🎯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Independent Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter

# === Initialize lists
all_scores = []
all_labels = []

# === Pairwise score computation for Session-Independent Verification (Subject + Finger)
for test_idx in range(len(test_lbp_features)):
    test_vec = test_lbp_features[test_idx]
    test_label = test_labels[test_idx]
    test_parts = test_label.split('_')
    test_id = f"{test_parts[0]}_{test_parts[1]}"  # ✅ Subject + Finger

    for train_idx in range(len(train_lbp_features)):
        train_vec = train_lbp_features[train_idx]
        train_label = train_labels[train_idx]
        train_parts = train_label.split('_')
        train_id = f"{train_parts[0]}_{train_parts[1]}"  # ✅ Subject + Finger

        # 🚫 Optional: Skip self-comparison
        if test_label == train_label:
            continue

        # 🔢 Similarity score: negative Manhattan distance
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # ✅ Ground truth: 1 if same subject + finger, 0 otherwise
        is_genuine = int(test_id == train_id)
        all_labels.append(is_genuine)

# === Normalize similarity scores to [0, 1]
scores = np.array(all_scores)
labels = np.array(all_labels)
scores = (scores - scores.min()) / (scores.max() - scores.min())

# === Optional: Apply SMOTE to balance classes
use_smote = False
if use_smote:
    smote = SMOTE(random_state=42)
    scores_2d = scores.reshape(-1, 1)
    scores_2d, labels = smote.fit_resample(scores_2d, labels)
    scores = scores_2d.ravel()
    print("🧪 After SMOTE label distribution:", Counter(labels))

# === Threshold Sweeping to Find Best F1 Score
best_f1 = best_thresh = best_prec = best_rec = 0

for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = precision
        best_rec = recall

# === Final Classification at Optimal Threshold
final_preds = (scores >= best_thresh).astype(int)
accuracy = accuracy_score(labels, final_preds)

# === Print Summary Report
print("🔍 Summary (Session-Independent Verification — Subject + Finger)")
print("📎 Feature: LBP((8,1),(16,1),(8,2)) + (2D)^2PCA")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {accuracy * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")
